In [ ]:
import pandas as pd
from utils import load_data
from tqdm import tqdm

path = "<path>"
projects = ["weaksatd", "big-vul", "devign"]

data_weaksatd = load_data("{}/{}/complete.csv".format(path, projects[0]))
data_big_vul = load_data("{}/{}/complete.csv".format(path, projects[1]))
data_devign = load_data("{}/{}/complete.csv".format(path, projects[2]))

In [ ]:
import os
path = "dataset/extracted_projects"
existing_folders = [dataset.lower() for dataset in os.listdir(path)]

def merge_f_lc(row):
    if isinstance(row["LeadingComment"], float) and math.isnan(row["LeadingComment"]):
        return row["Function"]

    return "\n".join([row["LeadingComment"],row["Function"]])


def merge_function_with_leading_comment(data, key = "FunctionWithLeadingComment"):
    tqdm.pandas(total=len(data), desc="Merging Function and Leading Comment...")
    data[key] = data.progress_apply(merge_f_lc, axis=1)

    return data

def create_folders_for_file(file_path):
    # Get the directory path from the file path
    directory_path = os.path.dirname(file_path)

    # Create the directory path if it doesn't exist
    os.makedirs(directory_path, exist_ok=True)

def write_into_files(data, project, dataset, path):
    data = data.progress_apply(merge_function_with_leading_comment)
    pbar = tqdm(total=len(data), desc="Writing the files for the project {} of the dataset {}".format(project, dataset))
    for idx, row in data.iterrows():
        filename = "{}/{}/{}_{}.c".format(path, project, dataset, idx).replace(" ", "_")
        create_folders_for_file(filename)
        data = row["FunctionWithComments"]

        with open(filename, 'w') as file:
                file.write(data.replace("\n\n", "\n"))
        pbar.update(1)

    pbar.close()

def extract_data(data, dataset):
    sub_datasets = data["Project"].unique()
    for sub_dataset in sub_datasets:
        folder = "{}_{}".format(sub_dataset, dataset).replace(" ", "_")
        if folder.lower() not in existing_folders and sub_dataset.lower() not in existing_folders:
            write_into_files(data[data["Project"] == sub_dataset], folder, sub_dataset, path)
        else:
            print("Skipping: {}".format(folder))

extract_data(data_big_vul, "big_vul")
extract_data(data_devign, "devign")
extract_data(data_weaksatd, "weaksatd")